## Cassidy's Notebook For Baseline Modelling

### Baseline Model (Majority Class Classifier): 
This notebook implements and evaluates the majority class classifier baseline model for predicting risky drinking behavior. 

The baseline establishes a simple benchmark that more advanced models (e.g., logistic regression, neural networks) must surpass to demonstrate meaningful predictive value.

Evaluation metrics referenced include:
- Accuracy, proportion of all predictions the model gets correct.
- Precision = TP / (TP + FP), how often the model’s positive predictions are actually correct.
- Recall = TP / (TP + FN), how many actual positive cases the model successfully detects.
- F1 Score, harmonic mean of precision and recall, balancing both types of errors.

In [7]:
# Imports
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Load data 
cleaned_X_train = pd.read_csv("cleaned_X_train.csv")
cleaned_y_train = pd.read_csv("cleaned_y_train.csv").squeeze()  # converts to Series

cleaned_X_val = pd.read_csv("cleaned_X_val.csv")
cleaned_y_val = pd.read_csv("cleaned_y_val.csv").squeeze()

cleaned_X_test = pd.read_csv("cleaned_X_test.csv")
cleaned_y_test = pd.read_csv("cleaned_y_test.csv").squeeze()

In [ ]:
# Set the majority class
majority_class = cleaned_y_train.mode()[0]
print("Majority class in TRAIN data:", majority_class)

# Create predeicitons
baseline_train_preds = np.full(len(cleaned_y_train), fill_value=majority_class)
baseline_val_preds   = np.full(len(cleaned_y_val), fill_value=majority_class)
baseline_test_preds  = np.full(len(cleaned_y_test), fill_value=majority_class)

Majority class in TRAIN data: False


In [13]:
# Get metrics
def get_metrics(data, y_true, y_pred):
    return {
        "Data": data,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1 Score": f1_score(y_true, y_pred, zero_division=0),
    }

metrics_train = get_metrics("Training", cleaned_y_train, baseline_train_preds)
metrics_val   = get_metrics("Validation", cleaned_y_val, baseline_val_preds)
metrics_test  = get_metrics("Test", cleaned_y_test, baseline_test_preds)

metrics_train, metrics_val, metrics_test

({'Data': 'Training',
  'Accuracy': 0.6228992466263763,
  'Precision': 0.0,
  'Recall': 0.0,
  'F1 Score': 0.0},
 {'Data': 'Validation',
  'Accuracy': 0.6228273464658169,
  'Precision': 0.0,
  'Recall': 0.0,
  'F1 Score': 0.0},
 {'Data': 'Test',
  'Accuracy': 0.6229476530809349,
  'Precision': 0.0,
  'Recall': 0.0,
  'F1 Score': 0.0})

In [ ]:
# Display metrics
summary_table = pd.DataFrame([{ **metrics_train}, {**metrics_val}, {**metrics_test }])
summary_table

,Data,Accuracy,Precision,Recall,F1 Score
0,Training,0.622899,0.0,0.0,0.0
1,Validation,0.622827,0.0,0.0,0.0
2,Test,0.622948,0.0,0.0,0.0


In [ ]:
# Specific subgroup columns to check performance for
subgroup_cols = ['_SEX', '_EDUCAG', '_SMOKER3', 'ACEDEPRS', 'ACEPRISN', 'ACEDIVRC', 'ACEPUNCH',
       'ACEHURT1', 'ACESWEAR', 'ACETOUCH']

subgroup_results = []

# Subgroup performance
for split_name, X, y in [
    ("Train", cleaned_X_train, cleaned_y_train),
    ("Validation", cleaned_X_val, cleaned_y_val),
    ("Test", cleaned_X_test, cleaned_y_test)
]:
    for col in subgroup_cols:
        if col not in X.columns:
            continue
        
        for group, idx in X.groupby(col).groups.items():
            y_true = y.loc[idx]
            y_pred = np.full(len(y_true), fill_value=majority_class)
            
            subgroup_results.append({
                "Subgroup Variable": col,
                "Group": group,
                "Count": len(y_true),
                **get_metrics(split_name, y_true, y_pred)
            })

# SDisplay subgroup metrics
subgroup_table = pd.DataFrame(subgroup_results)
subgroup_table


,Subgroup Variable,Group,Count,Data,Accuracy,Precision,Recall,F1 Score
0,_SEX,0.0,10863,Train,0.558317,0.0,0.0,0.0
1,_SEX,1.0,13295,Train,0.675668,0.0,0.0,0.0
2,_EDUCAG,1.0,1324,Train,0.796073,0.0,0.0,0.0
3,_EDUCAG,2.0,5936,Train,0.691712,0.0,0.0,0.0
4,_EDUCAG,3.0,6650,Train,0.645113,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...
82,ACESWEAR,1.0,215,Test,0.576744,0.0,0.0,0.0
83,ACESWEAR,2.0,1337,Test,0.572177,0.0,0.0,0.0
84,ACETOUCH,0.0,4582,Test,0.624836,0.0,0.0,0.0
85,ACETOUCH,1.0,203,Test,0.620690,0.0,0.0,0.0
